In [1]:
import json
import asyncio
import sys
import sqlite3

from pathlib import Path

# Walk up from CWD to find the project root (identified by pyproject.toml),
# so imports work regardless of where Jupyter is launched from.
def find_project_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    raise RuntimeError(f"Could not find project root from {start}")

project_root = find_project_root(Path().resolve())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from movie_ingestion.metadata_generation.inputs import MovieInputData
from movie_ingestion.metadata_generation.generators.plot_events import generate_plot_events
from movie_ingestion.metadata_generation.generators.reception import generate_reception
from movie_ingestion.metadata_generation.generators.plot_analysis import generate_plot_analysis
from movie_ingestion.metadata_generation.generators.viewer_experience import generate_viewer_experience
from movie_ingestion.metadata_generation.generators.watch_context import generate_watch_context
from movie_ingestion.metadata_generation.generators.narrative_techniques import generate_narrative_techniques
from movie_ingestion.metadata_generation.generators.production_keywords import generate_production_keywords
from movie_ingestion.metadata_generation.generators.source_of_inspiration import generate_source_of_inspiration
from implementation.llms.generic_methods import LLMProvider
from movie_ingestion.tracker import deserialize_imdb_row

/Users/michaelkeohane/Documents/movie-finder-rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ORIGINAL_SET_TMDB_IDS = [9377, 269149, 1584, 109445, 2493, 354912, 508965, 14160, 10674, 808, 13397, 76341, 85, 155, 245891, 1771, 569094, 299534, 11, 671, 120, 98, 27205, 603, 157336, 335984, 329, 329865, 493922, 694, 49018, 1034541, 176, 807, 496243, 419430, 1359, 550, 597, 13, 666277, 423, 11036, 1824, 25195, 216015, 392044, 545611, 22538, 37136]
MEDIUM_SPARSITY_TMDB_IDS = [329974, 1498832, 821937, 92, 160, 45739, 576560, 1383243, 1642210, 1639488]
HIGH_SPARSITY_TMDB_IDS = [270909, 493103, 64262, 1611977, 706910, 1297426, 35952, 158227, 215782, 1642486]

ALL_TMDB_IDS = ORIGINAL_SET_TMDB_IDS + MEDIUM_SPARSITY_TMDB_IDS + HIGH_SPARSITY_TMDB_IDS

# Open the tracker DB directly using the project root from the setup cell,
# avoiding reliance on CWD (init_db() uses a relative path).
db = sqlite3.connect(str(project_root / "ingestion_data" / "tracker.db"))
db.row_factory = sqlite3.Row

placeholders = ",".join("?" * len(ALL_TMDB_IDS))

# Fetch title + release_date from tmdb_data (the only fields tmdb_data has
# that we need; overview text lives in imdb_data.overview, not tmdb_data).
tmdb_rows: dict[int, sqlite3.Row] = {
    row["tmdb_id"]: row
    for row in db.execute(
        f"SELECT tmdb_id, title, release_date, maturity_rating "
        f"FROM tmdb_data WHERE tmdb_id IN ({placeholders})",
        ALL_TMDB_IDS,
    ).fetchall()
}

# Fetch all imdb_data columns; deserialize_imdb_row parses JSON array columns.
imdb_rows: dict[int, dict] = {
    row["tmdb_id"]: deserialize_imdb_row(row)
    for row in db.execute(
        f"SELECT * FROM imdb_data WHERE tmdb_id IN ({placeholders})",
        ALL_TMDB_IDS,
    ).fetchall()
}

db.close()


def build_movie_input(tmdb_id: int) -> MovieInputData | None:
    tmdb = tmdb_rows.get(tmdb_id)
    if tmdb is None:
        print(f"WARNING: tmdb_id {tmdb_id} not found in tmdb_data, skipping")
        return None
    imdb = imdb_rows.get(tmdb_id, {})

    release_date = tmdb["release_date"] or ""
    release_year = int(release_date[:4]) if release_date else None

    # Prefer IMDB maturity_rating; fall back to TMDB where IMDB has none.
    maturity_rating = imdb.get("maturity_rating") or tmdb["maturity_rating"] or ""

    return MovieInputData(
        tmdb_id=tmdb_id,
        title=tmdb["title"],
        release_year=release_year,
        overview=imdb.get("overview") or "",
        genres=imdb.get("genres") or [],
        plot_synopses=imdb.get("synopses") or [],      # imdb_data column is "synopses"
        plot_summaries=imdb.get("plot_summaries") or [],
        plot_keywords=imdb.get("plot_keywords") or [],
        overall_keywords=imdb.get("overall_keywords") or [],
        featured_reviews=imdb.get("featured_reviews") or [],
        reception_summary=imdb.get("reception_summary"),
        audience_reception_attributes=imdb.get("review_themes") or [],              # not stored in the DB
        maturity_rating=maturity_rating,
        maturity_reasoning=imdb.get("maturity_reasoning") or [],
        parental_guide_items=imdb.get("parental_guide_items") or [],
    )


movies = [m for tmdb_id in ALL_TMDB_IDS if (m := build_movie_input(tmdb_id)) is not None]
print(f"Built {len(movies)} MovieInputData objects from {len(ALL_TMDB_IDS)} requested IDs")

Built 70 MovieInputData objects from 70 requested IDs


In [3]:
from dataclasses import dataclass, field as dc_field


@dataclass(frozen=True)
class PlaygroundCandidate:
    """A model configuration to test in the playground.

    Lighter than EvaluationCandidate — no system_prompt or response_format
    since generator functions handle those internally.
    """
    label: str
    provider: LLMProvider
    model: str
    kwargs: dict = dc_field(default_factory=dict)


CANDIDATES = [
    PlaygroundCandidate(
        "gpt-5-mini-minimal-no-overview", LLMProvider.OPENAI, "gpt-5-mini",
        {"reasoning_effort": "minimal", "verbosity": "low"},
    ),
    # PlaygroundCandidate(
    #     "gpt-5-mini-low-new-prompt", LLMProvider.OPENAI, "gpt-5-mini",
    #     {"reasoning_effort": "low", "verbosity": "low"},
    # ),
    # PlaygroundCandidate(
    #     "gpt-5-mini-medium-new-prompt", LLMProvider.OPENAI, "gpt-5-mini",
    #     {"reasoning_effort": "medium", "verbosity": "low"},
    # ),
    # PlaygroundCandidate(
    #     "gpt-5.4-nano-medium", LLMProvider.OPENAI, "gpt-5.4-nano",
    #     {"reasoning_effort": "medium", "verbosity": "low"},
    # ),
    # PlaygroundCandidate(
    #     "gpt-5-nano", LLMProvider.OPENAI, "gpt-5-nano",
    #     {"reasoning_effort": "low", "verbosity": "low"},
    # ),
    # PlaygroundCandidate(
    #     "kimi-k2.5-no-thinking", LLMProvider.KIMI, "kimi-k2.5",
    #     # Thinking disabled — closest Kimi equivalent to gpt-5-mini reasoning_effort="low";
    #     # Kimi has no granular thinking budget, only on/off via enable_thinking
    #     {"enable_thinking": False},
    # ),
    # PlaygroundCandidate(
    #     "qwen3.5-flash", LLMProvider.ALIBABA, "qwen3.5-flash",
    #     # enable_thinking must be False — structured output not supported in thinking mode
    #     {"temperature": 0.2, "extra_body": {"enable_thinking": False}},
    # ),
    # PlaygroundCandidate(
    #     "gemini-2.5-flash-low", LLMProvider.GEMINI, "gemini-2.5-flash",
    #     # Small thinking budget: task involves cross-review interpretation, not standardized extraction
    #     {"temperature": 0.2, "thinking_config": {"thinking_budget": 1024}},
    # ),
    # PlaygroundCandidate(
    #     "gemini-2.5-flash-medium", LLMProvider.GEMINI, "gemini-2.5-flash",
    #     # Small thinking budget: task involves cross-review interpretation, not standardized extraction
    #     {"temperature": 0.2, "thinking_config": {"thinking_budget": 2048}},
    # ),
    # PlaygroundCandidate(
    #     "gemini-2.5-flash-lite", LLMProvider.GEMINI, "gemini-2.5-flash-lite",
    #     # Flash Lite does not support thinking configuration
    #     {"temperature": 0.2},
    # ),
    # PlaygroundCandidate(
    #     "gpt-oss-120b-high", LLMProvider.GROQ, "openai/gpt-oss-120b",
    #     {"temperature": 0.2, "reasoning_effort": "high", "include_reasoning": False},
    # ),
    # PlaygroundCandidate(
    #     "llama-4-scout", LLMProvider.GROQ, "meta-llama/llama-4-scout-17b-16e-instruct",
    #     {"temperature": 0.2},
    # ),
]

print(f"Defined {len(CANDIDATES)} candidates")

Defined 1 candidates


In [4]:
from pydantic import BaseModel


def print_all_fields(result: BaseModel) -> None:
    """Print every field of a Pydantic result on its own line.

    Solves the problem where __str__() omits certain fields
    (e.g., ReceptionOutput omits review_insights_brief).
    """
    for field_name in type(result).model_fields:
        value = getattr(result, field_name)
        if isinstance(value, list):
            print(f"  {field_name}:")
            for item in value:
                print(f"    - {item}")
        elif isinstance(value, BaseModel):
            print(f"  {field_name}:")
            for sub_field in type(value).model_fields:
                sub_value = getattr(value, sub_field)
                print(f"    {sub_field}: {sub_value}")
        else:
            print(f"  {field_name}: {value}")


async def run_candidates(movie: MovieInputData, generator_fn, extra_kwargs: dict | None = None) -> None:
    """Run all candidates through a generator function, printing results and token usage.

    Args:
        movie: The movie input data to test with.
        generator_fn: An async generator function (e.g., generate_reception).
        extra_kwargs: Additional keyword args passed to the generator before
            candidate kwargs (e.g., plot_synopsis, review_insights_brief).
    """
    if extra_kwargs is None:
        extra_kwargs = {}

    for candidate in CANDIDATES:
        print(f"\n{'─' * 60}")
        print(f"Model: {candidate.label}")
        try:
            result, token_usage = await generator_fn(
                movie,
                **extra_kwargs,
                provider=candidate.provider,
                model=candidate.model,
                **candidate.kwargs,
            )
            print_all_fields(result)
            print(f"  token_usage: {token_usage}")
        except Exception as e:
            print(f"  ERROR: {e}")

In [5]:
MODEL_PRICING: dict[str, tuple[float, float]] = {
    # (input_price_per_M, output_price_per_M)
    "qwen3.5-flash":                                   (0.10, 0.40),
    "gemini-2.5-flash":                                (0.30, 2.50),
    "gemini-2.5-flash-lite":                           (0.10, 0.40),
    "gpt-5-mini":                                      (0.25, 2.00),
    "gpt-5-nano":                                      (0.05, 0.40),
    "gpt-5.4-nano":                                    (0.20, 1.25),
    "openai/gpt-oss-120b":                             (0.15, 0.60),
    "meta-llama/llama-4-scout-17b-16e-instruct":       (0.11, 0.34),
    "kimi-k2.5":                                       (0.60, 3.00),
}

In [6]:
# # ── Plot Events (Wave 1) ─────────────────────────────────────────────────────
# # Inputs: overview, plot_summaries, plot_synopses
# # Output: plot_summary (single field)

# import json as _json
# from movie_ingestion.metadata_generation.generators.plot_events import (
#     build_plot_events_prompts,
#     generate_plot_events,
# )

# # ── Evaluation groups by synopsis / plot-summary size (first 3 per group) ────
# # Synopsis path (condensation): synopsis >= 2500 chars
# SYNOPSIS_BARELY_ABOVE_1K = [47678, 55297, 26736]
# SYNOPSIS_MEDIUM_4K = [986965, 16092, 411685]
# SYNOPSIS_MASSIVE_9K = [10616, 389, 53865]

# # Synthesis path: synopsis < 2500 (demoted or absent)
# SYNOPSIS_DEMOTED_600_TO_1K = [265090, 30599, 326293]
# PLOT_SUMMARIES_BARELY_ABOVE_600 = [77570, 327308, 116844]
# PLOT_SUMMARIES_MEDIUM_4K = [63376, 110803, 249318]
# PLOT_SUMMARIES_MASSIVE_6K = [86721, 220297, 816510]

# EVAL_GROUPS = {
#     "synopsis_barely_above_1k": SYNOPSIS_BARELY_ABOVE_1K,
#     "synopsis_medium_4k": SYNOPSIS_MEDIUM_4K,
#     "synopsis_massive_9k": SYNOPSIS_MASSIVE_9K,
#     "synopsis_demoted_600_to_1k": SYNOPSIS_DEMOTED_600_TO_1K,
#     "plot_summaries_barely_above_600": PLOT_SUMMARIES_BARELY_ABOVE_600,
#     "plot_summaries_medium_4k": PLOT_SUMMARIES_MEDIUM_4K,
#     "plot_summaries_massive_6k": PLOT_SUMMARIES_MASSIVE_6K,
# }

# # Load MovieInputData for all evaluation IDs
# ALL_EVAL_IDS = [tid for group in EVAL_GROUPS.values() for tid in group]

# db = sqlite3.connect(str(project_root / "ingestion_data" / "tracker.db"))
# db.row_factory = sqlite3.Row

# placeholders = ",".join("?" * len(ALL_EVAL_IDS))
# eval_tmdb_rows = {
#     row["tmdb_id"]: row
#     for row in db.execute(
#         f"SELECT tmdb_id, title, release_date, maturity_rating "
#         f"FROM tmdb_data WHERE tmdb_id IN ({placeholders})",
#         ALL_EVAL_IDS,
#     ).fetchall()
# }
# eval_imdb_rows = {
#     row["tmdb_id"]: deserialize_imdb_row(row)
#     for row in db.execute(
#         f"SELECT * FROM imdb_data WHERE tmdb_id IN ({placeholders})",
#         ALL_EVAL_IDS,
#     ).fetchall()
# }
# db.close()

# def _build_eval_movie(tmdb_id: int) -> MovieInputData | None:
#     tmdb = eval_tmdb_rows.get(tmdb_id)
#     if tmdb is None:
#         print(f"WARNING: tmdb_id {tmdb_id} not found in tmdb_data, skipping")
#         return None
#     imdb = eval_imdb_rows.get(tmdb_id, {})
#     release_date = tmdb["release_date"] or ""
#     release_year = int(release_date[:4]) if release_date else None
#     maturity_rating = imdb.get("maturity_rating") or tmdb["maturity_rating"] or ""
#     return MovieInputData(
#         tmdb_id=tmdb_id,
#         title=tmdb["title"],
#         release_year=release_year,
#         overview=imdb.get("overview") or "",
#         genres=imdb.get("genres") or [],
#         plot_synopses=imdb.get("synopses") or [],
#         plot_summaries=imdb.get("plot_summaries") or [],
#         plot_keywords=imdb.get("plot_keywords") or [],
#         overall_keywords=imdb.get("overall_keywords") or [],
#         featured_reviews=imdb.get("featured_reviews") or [],
#         reception_summary=imdb.get("reception_summary"),
#         audience_reception_attributes=imdb.get("review_themes") or [],
#         maturity_rating=maturity_rating,
#         maturity_reasoning=imdb.get("maturity_reasoning") or [],
#         parental_guide_items=imdb.get("parental_guide_items") or [],
#     )

# eval_movies: dict[int, MovieInputData] = {}
# for tid in ALL_EVAL_IDS:
#     m = _build_eval_movie(tid)
#     if m is not None:
#         eval_movies[tid] = m
# print(f"Built {len(eval_movies)} MovieInputData objects for evaluation")

# # ── Filter candidates: exclude Gemini models ────────────────────────────────
# non_gemini_candidates = [c for c in CANDIDATES if c.provider != LLMProvider.GEMINI]
# print(f"Using {len(non_gemini_candidates)} candidates (excluding Gemini):")
# for c in non_gemini_candidates:
#     print(f"  - {c.label} ({c.model})")


# def _compute_cost(model: str, input_tokens: int, output_tokens: int) -> float | None:
#     """Compute generation cost in USD from MODEL_PRICING."""
#     pricing = MODEL_PRICING.get(model)
#     if pricing is None:
#         return None
#     input_price, output_price = pricing
#     return (input_tokens * input_price + output_tokens * output_price) / 1_000_000


# async def _generate_for_candidate(
#     movie: MovieInputData,
#     candidate: "PlaygroundCandidate",
# ) -> dict:
#     """Generate plot events for one (movie, candidate) pair. Returns a result dict."""
#     try:
#         result, token_usage = await generate_plot_events(
#             movie,
#             provider=candidate.provider,
#             model=candidate.model,
#             **candidate.kwargs,
#         )
#         cost = _compute_cost(
#             token_usage.model, token_usage.input_tokens, token_usage.output_tokens,
#         )
#         cost_str = f"${cost:.6f}" if cost is not None else "n/a"
#         print(
#             f"  ✓ {candidate.label}: "
#             f"{token_usage.input_tokens} in / {token_usage.output_tokens} out, "
#             f"cost={cost_str}"
#         )
#         return {
#             "candidate": candidate.label,
#             "model": candidate.model,
#             "result": result.model_dump(),
#             "input_tokens": token_usage.input_tokens,
#             "output_tokens": token_usage.output_tokens,
#             "cost_usd": cost,
#         }
#     except Exception as e:
#         print(f"  ✗ {candidate.label}: {type(e).__name__}: {e}")
#         return {
#             "candidate": candidate.label,
#             "model": candidate.model,
#             "result": None,
#             "error": f"{type(e).__name__}: {e}",
#             "input_tokens": None,
#             "output_tokens": None,
#             "cost_usd": None,
#         }


# # ── Generate per-movie, all candidates concurrently ─────────────────────────

# eval_dir = project_root / "movie_ingestion" / "metadata_generation" / "evaluation_data"
# eval_dir.mkdir(parents=True, exist_ok=True)

# total_movies = len(eval_movies)
# for movie_idx, (tmdb_id, movie) in enumerate(eval_movies.items(), 1):
#     title = movie.title_with_year()
#     user_prompt, _ = build_plot_events_prompts(movie)

#     print(f"\n[{movie_idx}/{total_movies}] {title} (tmdb_id={tmdb_id})")

#     # Fire all candidates concurrently — each hits a different provider,
#     # so concurrent calls spread rate-limit pressure across providers.
#     candidate_results = await asyncio.gather(*[
#         _generate_for_candidate(movie, c) for c in non_gemini_candidates
#     ])

#     # Write per-movie JSON file
#     movie_output = {
#         "tmdb_id": tmdb_id,
#         "title": title,
#         "user_prompt": user_prompt,
#         "candidate_results": list(candidate_results),
#     }
#     out_path = eval_dir / f"plot_events_{tmdb_id}.json"
#     out_path.write_text(_json.dumps(movie_output, indent=2, ensure_ascii=False))
#     print(f"  Saved → {out_path.name}")

# print(f"\nDone: {total_movies} movies × {len(non_gemini_candidates)} candidates")

In [7]:
# ── Reception (Wave 1) ───────────────────────────────────────────────────────
# Inputs: reception_summary, audience_reception_attributes, featured_reviews
# Buckets: 6 groups by combined_top5_chars (review richness)

import json as _json
from movie_ingestion.metadata_generation.generators.reception import (
    build_reception_user_prompt,
    generate_reception,
)

# ── Load bucket IDs from reception_eval_buckets.json ─────────────────────────
_buckets_path = project_root / "ingestion_data" / "reception_eval_buckets.json"
with open(_buckets_path) as _f:
    _buckets = _json.load(_f)

ULTRA_THIN_0_1K = [s["tmdb_id"] for s in _buckets["ultra_thin_0_1000"]["samples"]]
VERY_THIN_1K_2500 = [s["tmdb_id"] for s in _buckets["very_thin_1000_2500"]["samples"]]
THIN_2500_5K = [s["tmdb_id"] for s in _buckets["thin_2500_5000"]["samples"]]
MODERATE_5K_7500 = [s["tmdb_id"] for s in _buckets["moderate_5000_7500"]["samples"]]
RICH_7500_10500 = [s["tmdb_id"] for s in _buckets["rich_7500_10500"]["samples"]]
VERY_RICH_10500_PLUS = [s["tmdb_id"] for s in _buckets["very_rich_10500_plus"]["samples"]]

EVAL_GROUPS = {
    "ultra_thin_0_1k": ULTRA_THIN_0_1K,
    "very_thin_1k_2500": VERY_THIN_1K_2500,
    "thin_2500_5k": THIN_2500_5K,
    "moderate_5k_7500": MODERATE_5K_7500,
    "rich_7500_10500": RICH_7500_10500,
    "very_rich_10500_plus": VERY_RICH_10500_PLUS,
}

# Load MovieInputData for all evaluation IDs
ALL_EVAL_IDS = [tid for group in EVAL_GROUPS.values() for tid in group]

db = sqlite3.connect(str(project_root / "ingestion_data" / "tracker.db"))
db.row_factory = sqlite3.Row

placeholders = ",".join("?" * len(ALL_EVAL_IDS))
eval_tmdb_rows = {
    row["tmdb_id"]: row
    for row in db.execute(
        f"SELECT tmdb_id, title, release_date, maturity_rating "
        f"FROM tmdb_data WHERE tmdb_id IN ({placeholders})",
        ALL_EVAL_IDS,
    ).fetchall()
}
eval_imdb_rows = {
    row["tmdb_id"]: deserialize_imdb_row(row)
    for row in db.execute(
        f"SELECT * FROM imdb_data WHERE tmdb_id IN ({placeholders})",
        ALL_EVAL_IDS,
    ).fetchall()
}
db.close()

def _build_eval_movie(tmdb_id: int) -> MovieInputData | None:
    tmdb = eval_tmdb_rows.get(tmdb_id)
    if tmdb is None:
        print(f"WARNING: tmdb_id {tmdb_id} not found in tmdb_data, skipping")
        return None
    imdb = eval_imdb_rows.get(tmdb_id, {})
    release_date = tmdb["release_date"] or ""
    release_year = int(release_date[:4]) if release_date else None
    maturity_rating = imdb.get("maturity_rating") or tmdb["maturity_rating"] or ""
    return MovieInputData(
        tmdb_id=tmdb_id,
        title=tmdb["title"],
        release_year=release_year,
        overview=imdb.get("overview") or "",
        genres=imdb.get("genres") or [],
        plot_synopses=imdb.get("synopses") or [],
        plot_summaries=imdb.get("plot_summaries") or [],
        plot_keywords=imdb.get("plot_keywords") or [],
        overall_keywords=imdb.get("overall_keywords") or [],
        featured_reviews=imdb.get("featured_reviews") or [],
        reception_summary=imdb.get("reception_summary"),
        audience_reception_attributes=imdb.get("review_themes") or [],
        maturity_rating=maturity_rating,
        maturity_reasoning=imdb.get("maturity_reasoning") or [],
        parental_guide_items=imdb.get("parental_guide_items") or [],
    )

eval_movies: dict[int, MovieInputData] = {}
for tid in ALL_EVAL_IDS:
    m = _build_eval_movie(tid)
    if m is not None:
        eval_movies[tid] = m
print(f"Built {len(eval_movies)} MovieInputData objects for evaluation")

# ── Use first 4 candidates ──────────────────────────────────────────────────
reception_candidates = CANDIDATES
print(f"Using {len(reception_candidates)} candidates:")
for c in reception_candidates:
    print(f"  - {c.label} ({c.model})")


def _compute_cost(model: str, input_tokens: int, output_tokens: int) -> float | None:
    """Compute generation cost in USD from MODEL_PRICING."""
    pricing = MODEL_PRICING.get(model)
    if pricing is None:
        return None
    input_price, output_price = pricing
    return (input_tokens * input_price + output_tokens * output_price) / 1_000_000


async def _generate_for_candidate(
    movie: MovieInputData,
    candidate: "PlaygroundCandidate",
) -> dict:
    """Generate reception for one (movie, candidate) pair. Returns a result dict."""
    try:
        result, token_usage = await generate_reception(
            movie,
            provider=candidate.provider,
            model=candidate.model,
            **candidate.kwargs,
        )
        cost = _compute_cost(
            token_usage.model, token_usage.input_tokens, token_usage.output_tokens,
        )
        cost_str = f"${cost:.6f}" if cost is not None else "n/a"
        print(
            f"  ✓ {candidate.label}: "
            f"{token_usage.input_tokens} in / {token_usage.output_tokens} out, "
            f"cost={cost_str}"
        )
        return {
            "candidate": candidate.label,
            "model": candidate.model,
            "result": result.model_dump(),
            "input_tokens": token_usage.input_tokens,
            "output_tokens": token_usage.output_tokens,
            "cost_usd": cost,
        }
    except Exception as e:
        print(f"  ✗ {candidate.label}: {type(e).__name__}: {e}")
        return {
            "candidate": candidate.label,
            "model": candidate.model,
            "result": None,
            "error": f"{type(e).__name__}: {e}",
            "input_tokens": None,
            "output_tokens": None,
            "cost_usd": None,
        }


def _merge_candidate_results(
    existing_results: list[dict],
    new_results: list[dict],
) -> list[dict]:
    """Merge new candidate results into existing ones.

    If a candidate with the same label already exists, replace that entry.
    Otherwise append the new entry. Preserves ordering of existing entries.
    """
    # Index existing results by label for O(1) lookup
    merged = {r["candidate"]: r for r in existing_results}
    # Overwrite or insert new results
    for r in new_results:
        merged[r["candidate"]] = r
    # Preserve original order for existing entries, append new ones at end
    seen = set()
    ordered = []
    for r in existing_results:
        label = r["candidate"]
        ordered.append(merged[label])
        seen.add(label)
    for r in new_results:
        if r["candidate"] not in seen:
            ordered.append(merged[r["candidate"]])
            seen.add(r["candidate"])
    return ordered


# ── Generate per-movie, all candidates concurrently ─────────────────────────

eval_dir = project_root / "movie_ingestion" / "metadata_generation" / "evaluation_data"
eval_dir.mkdir(parents=True, exist_ok=True)

total_movies = len(eval_movies)
for movie_idx, (tmdb_id, movie) in enumerate(eval_movies.items(), 1):
    title = movie.title_with_year()
    user_prompt = build_reception_user_prompt(movie)

    print(f"\n[{movie_idx}/{total_movies}] {title} (tmdb_id={tmdb_id})")

    # Fire all candidates concurrently — each hits a different provider,
    # so concurrent calls spread rate-limit pressure across providers.
    candidate_results = await asyncio.gather(*[
        _generate_for_candidate(movie, c) for c in reception_candidates
    ])

    # Load existing file (if any) and merge new results into it
    out_path = eval_dir / f"reception_{tmdb_id}.json"
    if out_path.exists():
        existing_output = _json.loads(out_path.read_text())
        existing_candidates = existing_output.get("candidate_results", [])
        merged = _merge_candidate_results(existing_candidates, list(candidate_results))
    else:
        merged = list(candidate_results)

    movie_output = {
        "tmdb_id": tmdb_id,
        "title": title,
        "user_prompt": user_prompt,
        "candidate_results": merged,
    }
    out_path.write_text(_json.dumps(movie_output, indent=2, ensure_ascii=False))
    print(f"  Saved → {out_path.name} ({len(merged)} candidates)")

print(f"\nDone: {total_movies} movies × {len(reception_candidates)} candidates")

Built 36 MovieInputData objects for evaluation
Using 1 candidates:
  - gpt-5-mini-minimal-no-overview (gpt-5-mini)

[1/36] Persona (2024) (tmdb_id=1259735)
  ✓ gpt-5-mini-minimal-no-overview: 1888 in / 203 out, cost=$0.000878
  Saved → reception_1259735.json (6 candidates)

[2/36] Kaikki pelissä (1994) (tmdb_id=327145)
  ✓ gpt-5-mini-minimal-no-overview: 1871 in / 246 out, cost=$0.000960
  Saved → reception_327145.json (6 candidates)

[3/36] Hit Man File (2005) (tmdb_id=45499)
  ✓ gpt-5-mini-minimal-no-overview: 1817 in / 272 out, cost=$0.000998
  Saved → reception_45499.json (6 candidates)

[4/36] Chirodini Tumi Je Amar (2008) (tmdb_id=327665)
  ✓ gpt-5-mini-minimal-no-overview: 1890 in / 300 out, cost=$0.001073
  Saved → reception_327665.json (6 candidates)

[5/36] Sri Kanakamalakshmi Recording Dance Troupe (1987) (tmdb_id=307959)
  ✓ gpt-5-mini-minimal-no-overview: 1839 in / 311 out, cost=$0.001082
  Saved → reception_307959.json (6 candidates)

[6/36] Tum Mere Ho (1990) (tmdb_id=15

In [ ]:
# ── Plot Analysis (Wave 2) ───────────────────────────────────────────────────
# Inputs: genres, plot_keywords + wave1(plot_synopsis, review_insights_brief)

MOVIE_IDX = 65
movie = movies[MOVIE_IDX]

# Fetch wave1 results for this movie (pass absolute path since CWD may differ)
wave1 = get_wave1_results([movie.tmdb_id], db_path=project_root / "ingestion_data" / "tracker.db")
w1 = wave1.get(movie.tmdb_id, {})
plot_synopsis = w1["plot_events"].plot_summary if w1.get("plot_events") else None
review_insights_brief = w1["reception"].review_insights_brief if w1.get("reception") else None

print(f"Title:                  {movie.title_with_year()}")
print(f"Genres:                 {movie.genres}")
print(f"Plot keywords:          {len(movie.plot_keywords)} entries")
if movie.plot_keywords:
    print(f"  {', '.join(movie.plot_keywords[:15])}{'…' if len(movie.plot_keywords) > 15 else ''}")
print(f"Wave1 plot_synopsis:    {plot_synopsis[:200] + '…' if plot_synopsis else '(none)'}")
print(f"Wave1 review_insights:  {review_insights_brief[:200] + '…' if review_insights_brief else '(none)'}")

# await run_candidates(movie, generate_plot_analysis, {
#     "plot_synopsis": plot_synopsis,
#     "review_insights_brief": review_insights_brief,
# })

Title:                  The Senator (2024)
Genres:                 ['Thriller']
Plot keywords:          15 entries
  gay sex, male frontal nudity, gay, senator, male rear nudity, male nudity, gay relationship, gay interest, gay protagonist, politician, shower, closeted homosexual, brazil, secret relationship, older man younger man relationship
Wave1 plot_synopsis:    Renan, a young man, is engaged in a secret, illicit affair with an influential, closeted right-wing senator. Renan falls in love with Victor, a journalist who is determined to expose the senator's hyp…
Wave1 review_insights:  (none)


In [ ]:
# ── Plot Analysis: gpt-5.1-mini justification vs non-justification ───────────
from movie_ingestion.metadata_generation.prompts.plot_analysis import (
    SYSTEM_PROMPT as PA_PROMPT,
    SYSTEM_PROMPT_WITH_JUSTIFICATIONS as PA_PROMPT_JUST,
)
from movie_ingestion.metadata_generation.schemas import (
    PlotAnalysisOutput,
    PlotAnalysisWithJustificationsOutput,
)

VARIANTS = [
    ("gpt-5.1 (no justifications)", PA_PROMPT, PlotAnalysisOutput),
    ("gpt-5.1 (with justifications)", PA_PROMPT_JUST, PlotAnalysisWithJustificationsOutput),
]

for label, sys_prompt, resp_fmt in VARIANTS:
    print(f"\n{'─' * 60}")
    print(f"Model: {label}")
    try:
        result, token_usage = await generate_plot_analysis(
            movie,
            plot_synopsis=plot_synopsis,
            review_insights_brief=review_insights_brief,
            provider=LLMProvider.OPENAI,
            model="gpt-5-mini",
            system_prompt=sys_prompt,
            response_format=resp_fmt,
            reasoning_effort="low",
        )
        print_all_fields(result)
        print(f"  token_usage: {token_usage}")
    except Exception as e:
        print(f"  ERROR: {e}")


────────────────────────────────────────────────────────────
Model: gpt-5.1-mini (no justifications)
  core_concept_label: Closeted power exposed
  genre_signatures:
    - political thriller
    - romantic scandal drama
    - investigative exposé
  conflict_scale: community
  character_arcs:
    - emerging autonomy
    - moral reckoning
    - public disgrace
  themes_primary:
    - Power and sexual secrecy
    - Hypocrisy of public morality
    - Love entangled with exposure
  lessons_learned:
    - Truth invites accountability
    - Secrecy corrodes integrity
  generalized_plot_overview: A young man maintains a secret affair with an influential, closeted politician while an investigative journalist seeks to expose that politician's hypocrisy; the journalist falls for the young man, turning a political exposé into a personal dilemma. The investigation escalates, forcing the young man and the journalist to choose between secrecy and truth, and ultimately exposing the politician's dupli

In [ ]:
# ── Viewer Experience (Wave 2) ───────────────────────────────────────────────
# Inputs: genres, merged_keywords(), maturity_summary() + wave1(plot_synopsis, review_insights_brief)

MOVIE_IDX = 61
movie = movies[MOVIE_IDX]

# Fetch wave1 results for this movie (pass absolute path since CWD may differ)
wave1 = get_wave1_results([movie.tmdb_id], db_path=project_root / "ingestion_data" / "tracker.db")
w1 = wave1.get(movie.tmdb_id, {})
plot_synopsis = w1["plot_events"].plot_summary if w1.get("plot_events") else None
review_insights_brief = w1["reception"].review_insights_brief if w1.get("reception") else None

print(f"Title:                  {movie.title_with_year()}")
print(f"Genres:                 {movie.genres}")
merged = movie.merged_keywords()
print(f"Merged keywords:        {len(merged)} entries")
if merged:
    print(f"  {', '.join(merged[:15])}{'…' if len(merged) > 15 else ''}")
print(f"Maturity summary:       {movie.maturity_summary() or '(none)'}")
print(f"Wave1 plot_synopsis:    {plot_synopsis[:200] + '…' if plot_synopsis else '(none)'}")
print(f"Wave1 review_insights:  {review_insights_brief[:200] + '…' if review_insights_brief else '(none)'}")

# await run_candidates(movie, generate_viewer_experience, {
#     "plot_synopsis": plot_synopsis,
#     "review_insights_brief": review_insights_brief,
# })

Title:                  Kalakalappu 2 (2018)
Genres:                 ['Comedy', 'Drama']
Merged keywords:        4 entries
  money, tamil, comedy, drama
Maturity summary:       Moderate Sex & Nudity, Mild Violence & Gore
Wave1 plot_synopsis:    Two groups are engaged in a conflict, fighting over a laptop and a suitcase filled with money. The situation escalates and becomes more humorous with the arrival of Siva.…
Wave1 review_insights:  (none)


In [ ]:
# ── Viewer Experience: gpt-5.1-mini justification vs non-justification ───────
from movie_ingestion.metadata_generation.prompts.viewer_experience import (
    SYSTEM_PROMPT as VE_PROMPT,
    SYSTEM_PROMPT_WITH_JUSTIFICATIONS as VE_PROMPT_JUST,
)
from movie_ingestion.metadata_generation.schemas import (
    ViewerExperienceOutput,
    ViewerExperienceWithJustificationsOutput,
)

VARIANTS = [
    ("gpt-5-mini (no justifications)", VE_PROMPT, ViewerExperienceOutput),
    ("gpt-5-mini (with justifications)", VE_PROMPT_JUST, ViewerExperienceWithJustificationsOutput),
]

for label, sys_prompt, resp_fmt in VARIANTS:
    print(f"\n{'─' * 60}")
    print(f"Model: {label}")
    try:
        result, token_usage = await generate_viewer_experience(
            movie,
            plot_synopsis=plot_synopsis,
            review_insights_brief=review_insights_brief,
            provider=LLMProvider.OPENAI,
            model="gpt-5-mini",
            system_prompt=sys_prompt,
            response_format=resp_fmt,
            reasoning_effort="low",
        )
        print_all_fields(result)
        print(f"  token_usage: {token_usage}")
    except Exception as e:
        print(f"  ERROR: {e}")


────────────────────────────────────────────────────────────
Model: gpt-5-mini (no justifications)
  emotional_palette:
    terms: ['lighthearted', 'goofy', 'laugh out loud', 'fast-paced fun', 'feel-good comedy', 'slapstick humor', 'quirky', 'cheeky']
    negations: ['not depressing', 'not heavy', 'no tearjerker', 'not slow']
  tension_adrenaline:
    terms: ['low stakes tension', 'comedic chaos', 'mildly frantic', 'playful scramble', 'occasionally frantic']
    negations: ['not too intense', 'no white knuckle', 'not anxiety inducing', 'not high adrenaline']
  tone_self_seriousness:
    terms: ['silly and playful', 'not serious', 'irreverent', 'broad comedy', 'ensemble hijinks']
    negations: ['not earnest drama', 'no pretension', 'not dark', 'not mean spirited']
  cognitive_complexity:
    terms: ['easy to follow', 'lightweight', 'digestible', 'straightforward plot', 'fast moving']
    negations: ['not confusing', 'not hard to follow', 'no deep thinking', 'not draining']
  disturban

In [ ]:
# ── Watch Context (Wave 2) ───────────────────────────────────────────────────
# Inputs: genres, merged_keywords(), maturity_summary() + wave1(review_insights_brief only, NO plot)

MOVIE_IDX = 0
movie = movies[MOVIE_IDX]

# Fetch wave1 results — watch_context deliberately receives NO plot_synopsis
wave1 = get_wave1_results([movie.tmdb_id], db_path=project_root / "ingestion_data" / "tracker.db")
w1 = wave1.get(movie.tmdb_id, {})
review_insights_brief = w1["reception"].review_insights_brief if w1.get("reception") else None

print(f"Title:                  {movie.title_with_year()}")
print(f"Genres:                 {movie.genres}")
merged = movie.merged_keywords()
print(f"Merged keywords:        {len(merged)} entries")
if merged:
    print(f"  {', '.join(merged[:15])}{'…' if len(merged) > 15 else ''}")
print(f"Maturity summary:       {movie.maturity_summary() or '(none)'}")
print(f"Wave1 review_insights:  {review_insights_brief[:200] + '…' if review_insights_brief else '(none)'}")

# await run_candidates(movie, generate_watch_context, {
#     "review_insights_brief": review_insights_brief,
# })

Title:                  Ferris Bueller's Day Off (1986)
Genres:                 ['Comedy']
Merged keywords:        9 entries
  high school, breaking the fourth wall, truancy, sibling rivalry, anti hero, buddy comedy, satire, teen comedy, comedy
Maturity summary:       PG-13 — Mild Sex & Nudity, Mild Violence & Gore, Moderate Profanity, Mild Alcohol, Drugs & Smoking
Wave1 review_insights:  (none)


In [ ]:
# ── Watch Context: gpt-5.1-mini justification vs non-justification ───────────
from movie_ingestion.metadata_generation.prompts.watch_context import (
    SYSTEM_PROMPT as WC_PROMPT,
    SYSTEM_PROMPT_WITH_JUSTIFICATIONS as WC_PROMPT_JUST,
)
from movie_ingestion.metadata_generation.schemas import (
    WatchContextOutput,
    WatchContextWithJustificationsOutput,
)

VARIANTS = [
    ("gpt-5-mini (no justifications)", WC_PROMPT, WatchContextOutput),
    ("gpt-5-mini (with justifications)", WC_PROMPT_JUST, WatchContextWithJustificationsOutput),
]

for label, sys_prompt, resp_fmt in VARIANTS:
    print(f"\n{'─' * 60}")
    print(f"Model: {label}")
    try:
        result, token_usage = await generate_watch_context(
            movie,
            review_insights_brief=review_insights_brief,
            provider=LLMProvider.OPENAI,
            model="gpt-5-mini",
            system_prompt=sys_prompt,
            response_format=resp_fmt,
            reasoning_effort="medium",
        )
        print_all_fields(result)
        print(f"  token_usage: {token_usage}")
    except Exception as e:
        print(f"  ERROR: {e}")


────────────────────────────────────────────────────────────
Model: gpt-5-mini (no justifications)
  self_experience_motivations:
    terms: ['feel rebellious', 'carefree escape', 'mood boosting comedy', 'nostalgic 80s vibes', 'laugh out loud', 'high school throwback']
  external_motivations:
    terms: ['teen comedy staple', 'entertain friends', 'pop-culture reference fodder']
  key_movie_feature_draws:
    terms: ['breaking the fourth wall', 'catchy soundtrack', 'snappy dialogue']
  watch_scenarios:
    terms: ['movie night with friends', 'lazy weekend watch', 'teen sleepover', 'nostalgic throwback night', 'light date night']
  token_usage: TokenUsage(input_tokens=1905, output_tokens=1162, model='gpt-5-mini')

────────────────────────────────────────────────────────────
Model: gpt-5-mini (with justifications)
  self_experience_motivations:
    justification: Phrases reflect the film's upbeat teen-comedy tone, truancy/rebellion themes, and light PG-13 content (genres + merged_keyword

In [ ]:
# ── Narrative Techniques (Wave 2) ────────────────────────────────────────────
# Inputs: genres, overall_keywords (NOT merged) + wave1(plot_synopsis, review_insights_brief)

MOVIE_IDX = 0
movie = movies[MOVIE_IDX]

# Fetch wave1 results for this movie (pass absolute path since CWD may differ)
wave1 = get_wave1_results([movie.tmdb_id], db_path=project_root / "ingestion_data" / "tracker.db")
w1 = wave1.get(movie.tmdb_id, {})
plot_synopsis = w1["plot_events"].plot_summary if w1.get("plot_events") else None
review_insights_brief = w1["reception"].review_insights_brief if w1.get("reception") else None

print(f"Title:                  {movie.title_with_year()}")
print(f"Genres:                 {movie.genres}")
print(f"Overall keywords:       {len(movie.overall_keywords)} entries")
if movie.overall_keywords:
    print(f"  {', '.join(movie.overall_keywords[:15])}{'…' if len(movie.overall_keywords) > 15 else ''}")
print(f"Wave1 plot_synopsis:    {plot_synopsis[:200] + '…' if plot_synopsis else '(none)'}")
print(f"Wave1 review_insights:  {review_insights_brief[:200] + '…' if review_insights_brief else '(none)'}")

# await run_candidates(movie, generate_narrative_techniques, {
#     "plot_synopsis": plot_synopsis,
#     "review_insights_brief": review_insights_brief,
# })

Title:                  Ferris Bueller's Day Off (1986)
Genres:                 ['Comedy']
Overall keywords:       4 entries
  Buddy Comedy, Satire, Teen Comedy, Comedy
Wave1 plot_synopsis:    High school senior Ferris Bueller fakes a serious illness to skip school for a day. He convinces his girlfriend, Sloane, and his best friend, Cameron, to join him for an adventure in Chicago. Ferris o…
Wave1 review_insights:  (none)


In [ ]:
# ── Narrative Techniques: gpt-5.1-mini justification vs non-justification ────
from movie_ingestion.metadata_generation.prompts.narrative_techniques import (
    SYSTEM_PROMPT as NT_PROMPT,
    SYSTEM_PROMPT_WITH_JUSTIFICATIONS as NT_PROMPT_JUST,
)
from movie_ingestion.metadata_generation.schemas import (
    NarrativeTechniquesOutput,
    NarrativeTechniquesWithJustificationsOutput,
)

VARIANTS = [
    ("gpt-5-mini (no justifications)", NT_PROMPT, NarrativeTechniquesOutput),
    ("gpt-5-mini (with justifications)", NT_PROMPT_JUST, NarrativeTechniquesWithJustificationsOutput),
]

for label, sys_prompt, resp_fmt in VARIANTS:
    print(f"\n{'─' * 60}")
    print(f"Model: {label}")
    try:
        result, token_usage = await generate_narrative_techniques(
            movie,
            plot_synopsis=plot_synopsis,
            review_insights_brief=review_insights_brief,
            provider=LLMProvider.OPENAI,
            model="gpt-5-mini",
            system_prompt=sys_prompt,
            response_format=resp_fmt,
            reasoning_effort="medium",
        )
        print_all_fields(result)
        print(f"  token_usage: {token_usage}")
    except Exception as e:
        print(f"  ERROR: {e}")


────────────────────────────────────────────────────────────
Model: gpt-5-mini (no justifications)
  pov_perspective:
    terms: ['first-person pov', 'direct-to-camera address']
  narrative_delivery:
    terms: ['linear chronology', 'one-day timeframe']
  narrative_archetype:
    terms: ['buddy romp']
  information_control:
    terms: ['dramatic irony', 'misdirection']
  characterization_methods:
    terms: ["show-don't-tell actions", 'character foil contrast']
  character_arcs:
    terms: ['coming-of-age arc', 'flat arc']
  audience_character_perception:
    terms: ['lovable rogue', 'sympathetic insecure friend', 'bumbling antagonist']
  conflict_stakes_design:
    terms: ['ticking clock deadline', 'escalation ladder']
  thematic_delivery:
    terms: ['contrast pairs', 'subtextual scenes']
  meta_techniques:
    terms: ['fourth-wall breaks', 'narrator commentary']
  additional_plot_devices:
    terms: ['montage sequences', 'episodic setpiece structure']
  token_usage: TokenUsage(inpu

In [ ]:
# ── Production Keywords (Wave 2) ─────────────────────────────────────────────
# Inputs: merged_keywords() — NO wave1 dependencies (classification task only)

MOVIE_IDX = 0
movie = movies[MOVIE_IDX]

merged = movie.merged_keywords()
print(f"Title:            {movie.title_with_year()}")
print(f"Merged keywords:  {len(merged)} entries")
if merged:
    print(f"  {', '.join(merged[:20])}{'…' if len(merged) > 20 else ''}")

# await run_candidates(movie, generate_production_keywords)

Title:            Ferris Bueller's Day Off (1986)
Merged keywords:  9 entries
  high school, breaking the fourth wall, truancy, sibling rivalry, anti hero, buddy comedy, satire, teen comedy, comedy


In [ ]:
# ── Production Keywords: gpt-5.1-mini justification vs non-justification ─────
from movie_ingestion.metadata_generation.prompts.production_keywords import (
    SYSTEM_PROMPT as PK_PROMPT,
    SYSTEM_PROMPT_WITH_JUSTIFICATIONS as PK_PROMPT_JUST,
)
from movie_ingestion.metadata_generation.schemas import (
    ProductionKeywordsOutput,
    ProductionKeywordsWithJustificationsOutput,
)

VARIANTS = [
    ("gpt-5-mini (no justifications)", PK_PROMPT, ProductionKeywordsOutput),
    ("gpt-5-mini (with justifications)", PK_PROMPT_JUST, ProductionKeywordsWithJustificationsOutput),
]

for label, sys_prompt, resp_fmt in VARIANTS:
    print(f"\n{'─' * 60}")
    print(f"Model: {label}")
    try:
        result, token_usage = await generate_production_keywords(
            movie,
            provider=LLMProvider.OPENAI,
            model="gpt-5-mini",
            system_prompt=sys_prompt,
            response_format=resp_fmt,
            reasoning_effort="low",
        )
        print_all_fields(result)
        print(f"  token_usage: {token_usage}")
    except Exception as e:
        print(f"  ERROR: {e}")


────────────────────────────────────────────────────────────
Model: gpt-5-mini (no justifications)
  terms:
    - breaking the fourth wall
  token_usage: TokenUsage(input_tokens=461, output_tokens=215, model='gpt-5-mini')

────────────────────────────────────────────────────────────
Model: gpt-5-mini (with justifications)
  justification: "breaking the fourth wall" is a filmmaking technique describing a production choice used in the film, whereas the other keywords describe plot, character types, or genres.
  terms:
    - breaking the fourth wall
  token_usage: TokenUsage(input_tokens=548, output_tokens=189, model='gpt-5-mini')


In [ ]:
# ── Source of Inspiration (Wave 2) ───────────────────────────────────────────
# Inputs: merged_keywords() + wave1(plot_synopsis, review_insights_brief)
# NOTE: This is the ONLY generation where parametric knowledge is allowed.

MOVIE_IDX = 41
movie = movies[MOVIE_IDX]

# Fetch wave1 results for this movie (pass absolute path since CWD may differ)
wave1 = get_wave1_results([movie.tmdb_id], db_path=project_root / "ingestion_data" / "tracker.db")
w1 = wave1.get(movie.tmdb_id, {})
plot_synopsis = w1["plot_events"].plot_summary if w1.get("plot_events") else None
review_insights_brief = w1["reception"].review_insights_brief if w1.get("reception") else None

merged = movie.merged_keywords()
print(f"Title:                  {movie.title_with_year()}")
print(f"Merged keywords:        {len(merged)} entries")
if merged:
    print(f"  {', '.join(merged[:15])}{'…' if len(merged) > 15 else ''}")
print(f"Wave1 plot_synopsis:    {plot_synopsis[:200] + '…' if plot_synopsis else '(none)'}")
print(f"Wave1 review_insights:  {review_insights_brief[:200] + '…' if review_insights_brief else '(none)'}")

# await run_candidates(movie, generate_source_of_inspiration, {
#     "plot_synopsis": plot_synopsis,
#     "review_insights_brief": review_insights_brief,
# })

Title:                  The Pianist (2002)
Merged keywords:        13 entries
  pianist, holocaust, hiding a jew, survival, based on autobiography, docudrama, epic, period drama, psychological drama, tragedy, war epic, biography, drama
Wave1 plot_synopsis:    In September 1939, Polish-Hebrew pianist Wladyslaw Szpilman is playing on the radio in Warsaw when the station is bombed during the Nazi invasion of Poland. After escaping the initial bombing, Szpilma…
Wave1 review_insights:  (none)


In [ ]:
# ── Source of Inspiration: gpt-5.1-mini justification vs non-justification ───
from movie_ingestion.metadata_generation.prompts.source_of_inspiration import (
    SYSTEM_PROMPT as SOI_PROMPT,
    SYSTEM_PROMPT_WITH_JUSTIFICATIONS as SOI_PROMPT_JUST,
)
from movie_ingestion.metadata_generation.schemas import (
    SourceOfInspirationOutput,
    SourceOfInspirationWithJustificationsOutput,
)

VARIANTS = [
    ("gpt-5-mini (no justifications)", SOI_PROMPT, SourceOfInspirationOutput),
    ("gpt-5-mini (with justifications)", SOI_PROMPT_JUST, SourceOfInspirationWithJustificationsOutput),
]

for label, sys_prompt, resp_fmt in VARIANTS:
    print(f"\n{'─' * 60}")
    print(f"Model: {label}")
    try:
        result, token_usage = await generate_source_of_inspiration(
            movie,
            plot_synopsis=plot_synopsis,
            review_insights_brief=review_insights_brief,
            provider=LLMProvider.OPENAI,
            model="gpt-5-mini",
            system_prompt=sys_prompt,
            response_format=resp_fmt,
            reasoning_effort="low",
        )
        print_all_fields(result)
        print(f"  token_usage: {token_usage}")
    except Exception as e:
        print(f"  ERROR: {e}")


────────────────────────────────────────────────────────────
Model: gpt-5-mini (no justifications)
  sources_of_inspiration:
    - based on an autobiography
    - based on a true story
  production_mediums:
    - live action
  token_usage: TokenUsage(input_tokens=1117, output_tokens=170, model='gpt-5-mini')

────────────────────────────────────────────────────────────
Model: gpt-5-mini (with justifications)
  justification: The merged keywords explicitly list "based on autobiography" and the plot matches the real-life story of Władysław Szpilman (Holocaust survival and postwar pianist), so the film adapts a true memoir. Visually the film is a live-action period drama with on-location/constructed wartime sets and real actors rather than animation.
  sources_of_inspiration:
    - based on a memoir/autobiography
    - based on a true story
  production_mediums:
    - live action
  token_usage: TokenUsage(input_tokens=1210, output_tokens=394, model='gpt-5-mini')


In [ ]:
input_cost_pm = 0.125
output_cost_pm = 1
input_tokens_per_movie = 2737
output_tokens_per_movie = 953
total_movie_count = 112_000


input_cost = input_tokens_per_movie * total_movie_count * (input_cost_pm / 1_000_000)
output_cost = output_tokens_per_movie * total_movie_count * (output_cost_pm / 1_000_000)

total_cost = input_cost + output_cost

print(f"Total cost: ${total_cost:.2f}")


Total cost: $145.05


In [ ]:
import json
import sqlite3
from pathlib import Path

db = sqlite3.connect(str(project_root / "ingestion_data" / "tracker.db"))
db.row_factory = sqlite3.Row

# Fetch all imdb_quality_passed movies that have an imdb_data row.
# synopses and plot_summaries are stored as JSON TEXT arrays.
rows = db.execute("""
    SELECT i.synopses, i.plot_summaries
    FROM movie_progress mp
    JOIN imdb_data i ON i.tmdb_id = mp.tmdb_id
    WHERE mp.status = 'imdb_quality_passed'
""").fetchall()
db.close()

total = len(rows)

has_synopsis   = []  # first-synopsis char lengths for movies with ≥1 synopsis
has_summaries  = []  # avg char length across all summaries for movies with ≥1 summary
has_neither    = 0

for row in rows:
    synopses      = json.loads(row["synopses"]      or "[]")
    plot_summaries = json.loads(row["plot_summaries"] or "[]")

    got_synopsis  = len(synopses) > 0
    got_summaries = len(plot_summaries) > 0

    if got_synopsis:
        has_synopsis.append(len(synopses[0]))

    if got_summaries:
        avg_len = sum(len(s) for s in plot_summaries) / len(plot_summaries)
        has_summaries.append(avg_len)

    if not got_synopsis and not got_summaries:
        has_neither += 1

print(f"Total imdb_quality_passed movies with imdb_data: {total:,}")
print()
print(f"Has ≥1 plot_synopsis:   {len(has_synopsis):,}  ({100*len(has_synopsis)/total:.1f}%)")
print(f"  Avg char len (first synopsis): {sum(has_synopsis)/len(has_synopsis):,.0f}")
print()
print(f"Has ≥1 plot_summary:    {len(has_summaries):,}  ({100*len(has_summaries)/total:.1f}%)")
print(f"  Avg char len (across all summaries): {sum(has_summaries)/len(has_summaries):,.0f}")
print()
print(f"Has neither:            {has_neither:,}  ({100*has_neither/total:.1f}%)")


Total imdb_quality_passed movies with imdb_data: 111,898

Has ≥1 plot_synopsis:   22,894  (20.5%)
  Avg char len (first synopsis): 4,689

Has ≥1 plot_summary:    51,328  (45.9%)
  Avg char len (across all summaries): 631

Has neither:            37,676  (33.7%)
